# Google Trends Recency Bias — reproduce in one run
**ArtaQuest Research** · [read the article](https://artaquest.org/research/?article=recency)

How much recent Google-Trends data should a model ignore? We **crop** the most recent stretch, refit on what remains (sweeping **0 → 6 years**), and score each crop by **pure in-sample R²** — no hold-out, no test set, just fitting power. **Runtime → Run all.**

### 1 · Load the open data
The 87 weekly series and the weekly sidereal ephemeris.

In [ ]:
import numpy as np, pandas as pd, json, urllib.request, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
B = 'https://artaquest.org/wp-content/uploads/research/'
ser = json.load(urllib.request.urlopen(B+'series.json'))
eph = pd.read_csv(B+'ephemeris_weekly.csv'); N=len(eph)
BODIES=['sun','moon','mercury','venus','mars','jupiter','saturn','uranus','neptune','pluto','chiron','node']
T={'sun':1.0,'moon':0.0748,'mercury':0.241,'venus':0.615,'mars':1.881,'jupiter':11.86,'saturn':29.46,
   'uranus':84.0,'neptune':164.8,'pluto':248.0,'chiron':50.0,'node':18.6}
REFS=np.arange(0,360,5)
des=[np.column_stack([np.sinc(np.deg2rad((eph[b].values-r+180)%360-180)/T[b]) for b in BODIES]) for r in REFS]
S=[np.array(v) for v in ser['series'].values()]; print(len(S),'series loaded')

### 2 · Crop → refit → score
For each crop (0–72 months) we refit every topic on the cropped series and take the **best-angle in-sample R²**, then the median across topics. If the recent tail is too noisy to fit, including it drags R² down; cropping it lifts R².

In [ ]:
WPM=365.25/7/12; MONTHS=list(range(0,13))+[15,18,24,30,36,48,60,72]; curve=[]
for mo in MONTHS:
    end=N-int(round(mo*WPM)); r2s=[]
    for y in S:
        yc=y[:end]; sst=((yc-yc.mean())**2).sum()
        if sst<1e-6: continue
        r2s.append(max(1-((yc-LinearRegression().fit(X[:end],yc).predict(X[:end]))**2).sum()/sst for X in des))
    curve.append((mo,float(np.median(r2s))))
    print(f'crop {mo:2d} mo: median in-sample R2 = {curve[-1][1]*100:.2f}%')
knee=next(c for c in curve if c[1]>=max(v for _,v in curve)-0.005)
print('knee (cheapest near-best crop):', knee[0],'months')

### 3 · One figure
Fitting power vs how much recent data is cropped. It climbs as the noisy recent year is dropped, knees at ~1 year, peaks ~30 months, then falls as too much history is discarded.

In [ ]:
mm=[c[0] for c in curve]; rr=[c[1]*100 for c in curve]
plt.figure(figsize=(11,3.6)); plt.plot(mm,rr,'-o',color='#E8B923',lw=2.4)
plt.axvline(knee[0],color='#2352E8',ls='--',lw=1.2,label=f'knee = {knee[0]} mo'); plt.axvline(12,color='gray',ls=':',lw=1,label='model crops 12 mo')
plt.xlabel('months of recent data cropped'); plt.ylabel('median in-sample R2 (%)')
plt.title('Fitting power vs recency crop (87 keywords)'); plt.legend(); plt.tight_layout(); plt.show()